# 05_5 — CatBoost Live Threshold Sweep

Notebook de diagnóstico para entender por qué `catboost_residual` gana offline pero hoy no produce señales live.

Aquí **no se reentrena nada**. Solo se toma el snapshot actual de `data/models/registry/live_scores.csv` y se vuelve a aplicar la política de señales cambiando:
- `buy_ev_threshold`
- `buy_roi_threshold`
- `max_price_yes`
- `max_days_to_end`
- y, de forma opcional, `p_yes_calibrated` vs `p_yes_raw`

Objetivo: distinguir si el cuello de botella es el modelo, la calibración o la política live.

In [1]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.scoring.signals import generate_signals

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})
sns.set_theme(style='whitegrid')

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
REGISTRY = ROOT / 'data' / 'models' / 'registry'
MODEL_NAME = 'catboost_residual'

live_df = pd.read_csv(REGISTRY / 'live_scores.csv')
df = live_df[live_df['model_name'] == MODEL_NAME].copy()

print('Registry:', REGISTRY)
print('Modelo:', MODEL_NAME)
print('Rows:', len(df))

Registry: /Users/andres/Documents/ITAM/octavo_semestre/mineria_y_analisis/proyecto_polymarket/data/models/registry
Modelo: catboost_residual
Rows: 529


## 1. Estado actual del snapshot live

In [2]:
base = df.copy()
print('Distribución actual de señales:')
print(base['signal'].value_counts())
print()

cols = ['price_yes', 'p_yes_raw', 'p_yes_calibrated', 'ev_per_share', 'expected_roi', 'expected_roi_capped', 'days_to_end']
display(base[cols].describe(percentiles=[0.1, 0.5, 0.9]).T[['mean', 'std', 'min', '10%', '50%', '90%', 'max']].round(4))

print('Valores únicos de p_yes_calibrated:', base['p_yes_calibrated'].nunique())
print(base['p_yes_calibrated'].value_counts().head(12))

Distribución actual de señales:
signal
HOLD    529
Name: count, dtype: int64



,mean,std,min,10%,50%,90%,max
price_yes,0.1163,0.2151,0.0005,0.0025,0.0130,0.3869,0.9985
p_yes_raw,0.0920,0.2006,0.0000,0.0000,0.0000,0.3106,0.9649
p_yes_calibrated,0.0857,0.2146,0.0000,0.0000,0.0000,0.2444,1.0000
ev_per_share,-0.0307,0.0438,-0.2087,-0.0965,-0.0095,-0.0025,0.0950
expected_roi,-0.8266,0.3400,-1.0000,-1.0000,-1.0000,-0.1296,0.1050
expected_roi_capped,-0.8266,0.3400,-1.0000,-1.0000,-1.0000,-0.1296,0.1050
days_to_end,245.5236,330.8720,0.0008,16.1606,78.1606,938.1606,938.1606


Valores únicos de p_yes_calibrated: 19
p_yes_calibrated
0.000000    405
0.068493     33
0.238255     25
1.000000     12
0.625000      9
0.244444      9
0.831325      8
0.398821      7
0.483721      4
0.323529      4
0.464912      3
0.679104      3
Name: count, dtype: int64


## 2. ¿Qué regla está bloqueando al modelo?

In [3]:
rules = {
    'min_price_yes >= 0.03': base['price_yes'] >= 0.03,
    'max_price_yes <= 0.85': base['price_yes'] <= 0.85,
    'max_days_to_end <= 180': base['days_to_end'] <= 180.0,
    'min_liquidity >= 1000': base['liquidity'] >= 1000.0,
    'min_volume_24h >= 100': base['volume_24h'] >= 100.0,
    'max_spread <= 0.10': base['spread'] <= 0.10,
    'buy_ev_threshold >= 0.03': base['ev_per_share'] >= 0.03,
    'buy_roi_threshold >= 0.10': base['expected_roi_capped'] >= 0.10,
}

rule_df = pd.DataFrame([
    {'regla': name, 'pasa': int(mask.sum()), 'falla': int((~mask).sum()), 'pass_rate': float(mask.mean())}
    for name, mask in rules.items()
]).sort_values('pass_rate', ascending=False)

display(rule_df.style.format({'pass_rate': '{:.1%}'}))

positive_ev = base[base['ev_per_share'] > 0].copy()
print(f'Mercados con EV positivo después de calibración: {len(positive_ev)}')
if not positive_ev.empty:
    diag_rows = []
    for name, mask in rules.items():
        submask = mask.loc[positive_ev.index]
        diag_rows.append({'regla': name, 'pasa_en_EV_positivo': int(submask.sum()), 'falla_en_EV_positivo': int((~submask).sum())})
    display(pd.DataFrame(diag_rows))

,regla,pasa,falla,pass_rate
3,min_liquidity >= 1000,529,0,100.0%
4,min_volume_24h >= 100,529,0,100.0%
5,max_spread <= 0.10,529,0,100.0%
1,max_price_yes <= 0.85,513,16,97.0%
2,max_days_to_end <= 180,344,185,65.0%
0,min_price_yes >= 0.03,217,312,41.0%
6,buy_ev_threshold >= 0.03,11,518,2.1%
7,buy_roi_threshold >= 0.10,3,526,0.6%


Mercados con EV positivo después de calibración: 20


,regla,pasa_en_EV_positivo,falla_en_EV_positivo
0,min_price_yes >= 0.03,20,0
1,max_price_yes <= 0.85,8,12
2,max_days_to_end <= 180,15,5
3,min_liquidity >= 1000,20,0
4,min_volume_24h >= 100,20,0
5,max_spread <= 0.10,20,0
6,buy_ev_threshold >= 0.03,11,9
7,buy_roi_threshold >= 0.10,3,17


## 3. `p_yes_raw` vs `p_yes_calibrated`

Si la calibración isotónica estuviera destruyendo la señal live, aquí deberíamos ver que usar `raw` ayuda a pasar thresholds. Lo probamos explícitamente.

In [4]:
def apply_policy(frame, prob_col='p_yes_calibrated', **signal_kwargs):
    work = frame.copy()
    work['p_yes_policy'] = work[prob_col]
    work['ev_per_share'] = work['p_yes_policy'] - work['price_yes']
    work['expected_roi'] = work['ev_per_share'] / work['price_yes'].clip(lower=1e-6)
    out = generate_signals(work, **signal_kwargs)
    actionable = out[out['signal'] != 'HOLD'].copy().sort_values('ev_per_share', ascending=False)
    return out, actionable

base_kwargs = dict(
    buy_ev_threshold=0.03,
    strong_buy_ev_threshold=0.07,
    buy_roi_threshold=0.10,
    strong_buy_roi_threshold=0.20,
    min_price_yes=0.03,
    max_price_yes=0.85,
    max_days_to_end=180.0,
    min_liquidity=1000.0,
    min_volume_24h=100.0,
    max_spread=0.10,
)

for prob_col in ['p_yes_calibrated', 'p_yes_raw']:
    scored, actionable = apply_policy(base, prob_col=prob_col, **base_kwargs)
    print(f'Policy with {prob_col}:')
    print(scored['signal'].value_counts().to_dict())
    print('positive EV count:', int((scored['ev_per_share'] > 0).sum()))
    print('actionable count:', len(actionable))
    print()

Policy with p_yes_calibrated:
{'HOLD': 529}
positive EV count: 20
actionable count: 0

Policy with p_yes_raw:
{'HOLD': 529}
positive EV count: 1
actionable count: 0



## 4. Escenarios concretos de política live

Primero probamos un conjunto corto de escenarios razonables antes de barrer thresholds finos.

In [5]:
scenarios = {
    'current_calibrated': dict(
        prob_col='p_yes_calibrated',
        buy_ev_threshold=0.03,
        strong_buy_ev_threshold=0.07,
        buy_roi_threshold=0.10,
        strong_buy_roi_threshold=0.20,
        min_price_yes=0.03,
        max_price_yes=0.85,
        max_days_to_end=180.0,
    ),
    'cal_relaxed_price_days': dict(
        prob_col='p_yes_calibrated',
        buy_ev_threshold=0.03,
        strong_buy_ev_threshold=0.07,
        buy_roi_threshold=0.10,
        strong_buy_roi_threshold=0.20,
        min_price_yes=0.03,
        max_price_yes=0.95,
        max_days_to_end=365.0,
    ),
    'cal_relaxed_thresholds': dict(
        prob_col='p_yes_calibrated',
        buy_ev_threshold=0.02,
        strong_buy_ev_threshold=0.05,
        buy_roi_threshold=0.05,
        strong_buy_roi_threshold=0.12,
        min_price_yes=0.03,
        max_price_yes=0.95,
        max_days_to_end=365.0,
    ),
    'cal_aggressive': dict(
        prob_col='p_yes_calibrated',
        buy_ev_threshold=0.01,
        strong_buy_ev_threshold=0.03,
        buy_roi_threshold=0.02,
        strong_buy_roi_threshold=0.08,
        min_price_yes=0.03,
        max_price_yes=0.99,
        max_days_to_end=365.0,
    ),
    'raw_current_rules': dict(
        prob_col='p_yes_raw',
        buy_ev_threshold=0.03,
        strong_buy_ev_threshold=0.07,
        buy_roi_threshold=0.10,
        strong_buy_roi_threshold=0.20,
        min_price_yes=0.03,
        max_price_yes=0.85,
        max_days_to_end=180.0,
    ),
    'raw_relaxed': dict(
        prob_col='p_yes_raw',
        buy_ev_threshold=0.02,
        strong_buy_ev_threshold=0.05,
        buy_roi_threshold=0.05,
        strong_buy_roi_threshold=0.12,
        min_price_yes=0.03,
        max_price_yes=0.95,
        max_days_to_end=365.0,
    ),
}

scenario_outputs = {}
rows = []
for name, params in scenarios.items():
    params = params.copy()
    prob_col = params.pop('prob_col')
    merged = {**base_kwargs, **params}
    scored, actionable = apply_policy(base, prob_col=prob_col, **merged)
    scenario_outputs[name] = {'scored': scored, 'actionable': actionable, 'prob_col': prob_col, 'params': params}
    rows.append({
        'scenario': name,
        'prob_col': prob_col,
        'buy_count': int((scored['signal'] == 'BUY').sum()),
        'strong_buy_count': int((scored['signal'] == 'STRONG BUY').sum()),
        'actionable_count': int((scored['signal'] != 'HOLD').sum()),
        'positive_ev_count': int((scored['ev_per_share'] > 0).sum()),
        'top10_avg_ev': float(actionable.head(10)['ev_per_share'].mean()) if len(actionable) else np.nan,
        'top10_avg_roi_capped': float(actionable.head(10)['expected_roi_capped'].mean()) if len(actionable) else np.nan,
        'max_ev': float(scored['ev_per_share'].max()),
    })

scenario_df = pd.DataFrame(rows).sort_values(['actionable_count', 'top10_avg_ev'], ascending=[False, False])
display(scenario_df.style.format({
    'top10_avg_ev': '{:.4f}',
    'top10_avg_roi_capped': '{:.4f}',
    'max_ev': '{:.4f}',
}))

,scenario,prob_col,buy_count,strong_buy_count,actionable_count,positive_ev_count,top10_avg_ev,top10_avg_roi_capped,max_ev
3,cal_aggressive,p_yes_calibrated,8,6,14,20,0.0649,0.0866,0.0950
2,cal_relaxed_thresholds,p_yes_calibrated,10,0,10,20,0.0649,0.0866,0.0950
1,cal_relaxed_price_days,p_yes_calibrated,3,0,3,20,0.0827,0.1041,0.0950
0,current_calibrated,p_yes_calibrated,0,0,0,20,nan,nan,0.0950
4,raw_current_rules,p_yes_raw,0,0,0,1,nan,nan,0.0023
5,raw_relaxed,p_yes_raw,0,0,0,1,nan,nan,0.0023


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(scenario_df['scenario'], scenario_df['actionable_count'], color='#1565C0', edgecolor='white')
ax.set_title('Señales accionables por escenario')
ax.set_ylabel('count')
ax.tick_params(axis='x', rotation=35)

ax = axes[1]
valid = scenario_df.dropna(subset=['top10_avg_ev'])
ax.bar(valid['scenario'], valid['top10_avg_ev'], color='#2E7D32', edgecolor='white')
ax.set_title('Top-10 avg EV entre accionables')
ax.set_ylabel('EV por share')
ax.tick_params(axis='x', rotation=35)
ax.axhline(0, color='black', lw=1)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_live_threshold_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_36081/1624904672.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Sweep fino de thresholds

Con `p_yes_calibrated`, `max_price_yes=0.95` y `max_days_to_end=365`, barrimos EV/ROI para ver cuántas señales emergen.

In [7]:
ev_grid = [0.01, 0.02, 0.03, 0.04, 0.05]
roi_grid = [0.02, 0.05, 0.08, 0.10, 0.12]
heat_rows = []
for ev_thr in ev_grid:
    for roi_thr in roi_grid:
        scored, actionable = apply_policy(
            base,
            prob_col='p_yes_calibrated',
            buy_ev_threshold=ev_thr,
            strong_buy_ev_threshold=max(ev_thr * 2, ev_thr + 0.02),
            buy_roi_threshold=roi_thr,
            strong_buy_roi_threshold=max(roi_thr * 2, roi_thr + 0.05),
            min_price_yes=0.03,
            max_price_yes=0.95,
            max_days_to_end=365.0,
            min_liquidity=1000.0,
            min_volume_24h=100.0,
            max_spread=0.10,
        )
        heat_rows.append({
            'buy_ev_threshold': ev_thr,
            'buy_roi_threshold': roi_thr,
            'actionable_count': int((scored['signal'] != 'HOLD').sum()),
            'top10_avg_ev': float(actionable.head(10)['ev_per_share'].mean()) if len(actionable) else np.nan,
            'top10_avg_roi_capped': float(actionable.head(10)['expected_roi_capped'].mean()) if len(actionable) else np.nan,
        })

heat_df = pd.DataFrame(heat_rows)
display(heat_df.sort_values(['actionable_count', 'top10_avg_ev'], ascending=[False, False]).head(15))

,buy_ev_threshold,buy_roi_threshold,actionable_count,top10_avg_ev,top10_avg_roi_capped
0,0.01,0.02,11,0.064900,0.086552
1,0.01,0.05,11,0.064900,0.086552
5,0.02,0.02,10,0.064900,0.086552
6,0.02,0.05,10,0.064900,0.086552
10,0.03,0.02,10,0.064900,0.086552
11,0.03,0.05,10,0.064900,0.086552
15,0.04,0.02,10,0.064900,0.086552
16,0.04,0.05,10,0.064900,0.086552
20,0.05,0.02,8,0.071125,0.091096
21,0.05,0.05,8,0.071125,0.091096


In [8]:
pivot_count = heat_df.pivot(index='buy_roi_threshold', columns='buy_ev_threshold', values='actionable_count')
pivot_ev = heat_df.pivot(index='buy_roi_threshold', columns='buy_ev_threshold', values='top10_avg_ev')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot_count, annot=True, fmt='.0f', cmap='Blues', ax=axes[0])
axes[0].set_title('Actionable count')
axes[0].set_xlabel('buy_ev_threshold')
axes[0].set_ylabel('buy_roi_threshold')

sns.heatmap(pivot_ev, annot=True, fmt='.3f', cmap='Greens', ax=axes[1])
axes[1].set_title('Top-10 avg EV')
axes[1].set_xlabel('buy_ev_threshold')
axes[1].set_ylabel('buy_roi_threshold')

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_live_threshold_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_36081/2486189827.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Top candidatos bajo un escenario recomendado

Tomamos un escenario intermedio, no el más agresivo: `p_yes_calibrated`, `max_price_yes=0.95`, `max_days_to_end=365`, `buy_ev_threshold=0.02`, `buy_roi_threshold=0.05`.

In [9]:
recommended_name = 'cal_relaxed_thresholds'
recommended = scenario_outputs[recommended_name]['actionable']
print('Escenario recomendado:', recommended_name)
print('Parámetros:', scenario_outputs[recommended_name]['params'])
print('Signals:', scenario_outputs[recommended_name]['scored']['signal'].value_counts().to_dict())
print()

cols = ['question', 'price_yes', 'p_yes_raw', 'p_yes_calibrated', 'ev_per_share', 'expected_roi_capped', 'days_to_end', 'signal']
if recommended.empty:
    print('Sigue sin señales accionables.')
else:
    display(recommended[cols].head(20).style.format({
        'price_yes': '{:.3f}',
        'p_yes_raw': '{:.3f}',
        'p_yes_calibrated': '{:.3f}',
        'ev_per_share': '{:.4f}',
        'expected_roi_capped': '{:.4f}',
        'days_to_end': '{:.1f}',
    }))

Escenario recomendado: cal_relaxed_thresholds
Parámetros: {'buy_ev_threshold': 0.02, 'strong_buy_ev_threshold': 0.05, 'buy_roi_threshold': 0.05, 'strong_buy_roi_threshold': 0.12, 'min_price_yes': 0.03, 'max_price_yes': 0.95, 'max_days_to_end': 365.0}
Signals: {'HOLD': 519, 'BUY': 10}



,question,price_yes,p_yes_raw,p_yes_calibrated,ev_per_share,expected_roi_capped,days_to_end,signal
1587,"Will Bitcoin reach $70,000 in April?",0.905,0.881,1.000,0.0950,0.1050,17.3,BUY
1588,Will Anthropic have the best AI model at the end of April 2026?,0.905,0.877,1.000,0.0950,0.1050,16.2,BUY
1589,Will the Virginia redistricting referendum pass?,0.915,0.863,1.000,0.0850,0.0929,7.2,BUY
1590,"Will Israel take military action in Gaza on April 5, 2026?",0.934,0.897,1.000,0.0660,0.0707,16.2,BUY
1591,Will Shai Gilgeous-Alexander win the 2025–2026 NBA MVP?,0.935,0.876,1.000,0.0650,0.0695,57.2,BUY
1592,Will Mojtaba Khamenei be head of state in Iran end of 2026?,0.567,0.493,0.625,0.0580,0.1023,261.2,BUY
1593,Will Bitcoin hit $60k or $80k first?,0.570,0.545,0.625,0.0550,0.0965,262.4,BUY
1594,Los Angeles Dodgers vs. Toronto Blue Jays,0.575,0.546,0.625,0.0500,0.0870,0.1,BUY
1595,Will the U.S. invade Iran before 2027?,0.585,0.522,0.625,0.0400,0.0684,261.2,BUY
1596,Iran x Israel/US conflict ends by June 30?,0.585,0.549,0.625,0.0400,0.0684,77.2,BUY


## 7. Lectura final

Este notebook debería ayudarte a contestar tres cosas:
1. si el problema live es el modelo o los thresholds
2. si usar `raw` ayuda o no
3. qué política relajada produce señales sin volverse absurda

In [10]:
best = scenario_df.iloc[0]
print('Resumen rápido:')
print(f"- Escenario con más señales: {best['scenario']} ({int(best['actionable_count'])} accionables).")
print('- Con las reglas actuales, CatBoost no genera señales live.')
print('- En este snapshot, relajar price cap / horizon ayuda más que cambiar raw vs calibrated.')
print('- Si quieres usar CatBoost live, hoy parece más razonable probar thresholds intermedios antes que reemplazar la calibración.')

Resumen rápido:
- Escenario con más señales: cal_aggressive (14 accionables).
- Con las reglas actuales, CatBoost no genera señales live.
- En este snapshot, relajar price cap / horizon ayuda más que cambiar raw vs calibrated.
- Si quieres usar CatBoost live, hoy parece más razonable probar thresholds intermedios antes que reemplazar la calibración.
